# HGCTM Revision — Phase 1B GLiM Lithology Audit

This notebook performs an independent **full-polygon lithology extraction** for the 1,335 deposits used in the submitted HGCTM-S analysis.

It does not refit HGCTM-S. It will:

1. Load the frozen 1,335-deposit model coordinates.
2. Extract GLiM/LiMW polygon attributes at every coordinate.
3. Preserve original polygon codes and all available source/quality fields.
4. Apply a predefined cautious mapping into the HGCTM-S lithology classes.
5. Compare GLiM with the original Macrostrat assignments.
6. Report recovery among the original 100 `unknown` and 375 `other` records.
7. Export three transparent lithology versions:
   - original Macrostrat;
   - GLiM-only;
   - Macrostrat with GLiM fallback.
8. Produce a ranked list and a stratified sample for manual host-rock validation.

## Interpretation

Macrostrat and GLiM describe mapped surface or map-unit lithology. Neither is assumed to be the true deposit-scale host rock. These outputs are intended for sensitivity analysis and to guide independent geological validation.

## Verified references

**Jens Hartmann and Nils Moosdorf.** “The new global lithological map database GLiM: A representation of rock properties at the Earth surface.” *Geochemistry, Geophysics, Geosystems* 13, Q12004 (2012). DOI: **10.1029/2012GC004370**.

**Jens Hartmann and Nils Moosdorf.** “Global Lithological Map Database v1.0 (gridded to 0.5° spatial resolution).” PANGAEA (2012). DOI: **10.1594/PANGAEA.788537**. The coarse gridded product is cited for provenance but is not used in this notebook.

## Kaggle setup

Attach:

1. The Stage 0 dataset or ZIP containing `covariates.csv` and `macrostrat_lithology.csv`.
2. Preferably, the full polygon archive `LiMW_GIS-2015.gdb.zip`.

If the polygon archive is not attached, keep `AUTO_DOWNLOAD_GIS = True` and enable Internet access in Kaggle. The original submitted notebook and Stage 0 files are never modified.

In [ ]:
# Install only packages that are missing.

import importlib.util
import subprocess
import sys

required = {
    "geopandas": "geopandas>=0.14",
    "pyogrio": "pyogrio>=0.8",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import re
import shutil
import zipfile

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import requests

from pyproj import Transformer
from sklearn.metrics import cohen_kappa_score
from tqdm.auto import tqdm

WORK = Path("/kaggle/working")
OUT = WORK / "phase1b_glim_audit"
CACHE = WORK / "phase1b_glim_cache"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# ---------------- USER SETTINGS ----------------

AUTO_DOWNLOAD_GIS = True

# Current full-polygon download linked from the University of Hamburg GLiM page.
GLIM_POLYGON_URL = (
    "https://www.dropbox.com/scl/fi/5v00i8op7a9brmn4qeg8b/"
    "LiMW_GIS-2015.gdb.zip"
    "?rlkey=89q1aj8zfpmiv82y41xe6ttmp&dl=1"
)

GLIM_VERSION_LABEL = "LiMW_GIS-2015 full polygon GIS linked from UHH GLiM page"

# Local tiled reading avoids loading all global polygons into memory.
TILE_DEGREES = 20.0
BBOX_PADDING_DEGREES = 0.05

# Normally leave these as None. Set only if automatic detection fails.
POLYGON_LAYER_OVERRIDES = None
LEVEL1_FIELD_OVERRIDES = None

MANUAL_VALIDATION_SAMPLE_N = 120
RANDOM_SEED = 20260617

print("pyogrio:", pyogrio.__version__)
print("OpenFileGDB driver:", pyogrio.list_drivers().get("OpenFileGDB"))


## 1. Locate and load the frozen Stage 0 files

In [ ]:
STAGE0_REQUIRED = {"covariates.csv", "macrostrat_lithology.csv"}

def find_stage0_dir():
    roots = [Path("/kaggle/input"), WORK]

    for root in roots:
        if not root.exists():
            continue
        for cov_path in root.rglob("covariates.csv"):
            folder = cov_path.parent
            present = {p.name for p in folder.iterdir() if p.is_file()}
            if STAGE0_REQUIRED.issubset(present):
                return folder

    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as zf:
                    names = {Path(name).name for name in zf.namelist()}
                    if STAGE0_REQUIRED.issubset(names):
                        target = CACHE / "stage0"
                        target.mkdir(parents=True, exist_ok=True)
                        zf.extractall(target)
                        return target
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        "Attach Hgctm_stage0.zip or the extracted Stage 0 Kaggle dataset."
    )

STAGE0 = find_stage0_dir()
cov = pd.read_csv(STAGE0 / "covariates.csv")
macro = pd.read_csv(STAGE0 / "macrostrat_lithology.csv")

required_cov_cols = {
    "Mindat_id", "Deposit_type", "age_category",
    "litho_class", "Latitude", "Longitude",
}
missing_cols = required_cov_cols - set(cov.columns)
if missing_cols:
    raise KeyError(f"covariates.csv is missing: {sorted(missing_cols)}")

if len(cov) != 1335:
    raise ValueError(f"Expected 1,335 model deposits, found {len(cov):,}.")

if cov["Mindat_id"].duplicated().any():
    raise ValueError("covariates.csv contains duplicated Mindat_id values.")

print("Stage 0 directory:", STAGE0)
print("Model deposits:", len(cov))
print("Macrostrat records:", len(macro))


In [ ]:
# Optional: attach deposit names and countries from the main Excel database.

def find_main_database():
    for root in [Path("/kaggle/input"), WORK]:
        if not root.exists():
            continue
        for path in root.rglob("*.xlsx"):
            try:
                preview = pd.read_excel(path, sheet_name=0, nrows=3)
                if {"Mindat_id", "Deposit_name"}.issubset(preview.columns):
                    return path
            except Exception:
                continue
    return None

main_db_path = find_main_database()
deposit_lookup = None

if main_db_path is not None:
    main_db = pd.read_excel(main_db_path, sheet_name=0)
    wanted = [
        column for column in [
            "Mindat_id", "Deposit_name", "Country",
            "Latitude", "Longitude", "Deposit_type", "Mindat_url",
        ]
        if column in main_db.columns
    ]
    deposit_lookup = main_db[wanted].drop_duplicates("Mindat_id").copy()
    print("Deposit names loaded from:", main_db_path)
else:
    print("Main Excel database not attached; Mindat_id and coordinates will be used.")


## 2. Locate or download the full polygon GIS

Accepted inputs:

- extracted `.gdb`;
- ZIP containing a `.gdb`;
- clearly named GLiM/LiMW `.gpkg`;
- clearly named GLiM/LiMW `.shp`.

The coarse 0.5° table is intentionally not accepted.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_with_progress(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print("Using cached download:", destination)
        return destination

    headers = {"User-Agent": "Mozilla/5.0 HGCTM-revision-audit/1.0"}
    print("Downloading full polygon GIS archive...")
    with requests.get(
        url,
        stream=True,
        timeout=(30, 300),
        allow_redirects=True,
        headers=headers,
    ) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        with open(destination, "wb") as handle, tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    progress.update(len(chunk))

    if not zipfile.is_zipfile(destination):
        raise RuntimeError(
            "The downloaded file is not a valid ZIP. Attach the official GIS archive manually."
        )
    return destination

def zip_contains_gdb(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return any(".gdb/" in name.lower() for name in zf.namelist())
    except zipfile.BadZipFile:
        return False

def extract_gis_zip(path):
    target = CACHE / "glim_polygon_source"
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(path) as zf:
        zf.extractall(target)

    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
    return target

def find_vector_source():
    roots = [Path("/kaggle/input"), WORK, CACHE]

    for root in roots:
        if root.exists():
            gdbs = list(root.rglob("*.gdb"))
            if gdbs:
                return sorted(gdbs, key=lambda p: len(str(p)))[0], None

    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob("*.zip"):
            if zip_contains_gdb(path):
                extracted = extract_gis_zip(path)
                gdbs = list(extracted.rglob("*.gdb"))
                if not gdbs:
                    raise FileNotFoundError("No .gdb found after extraction.")
                return gdbs[0], path

    for root in roots:
        if not root.exists():
            continue
        candidates = [
            path
            for path in list(root.rglob("*.gpkg")) + list(root.rglob("*.shp"))
            if any(token in path.name.lower() for token in ["glim", "limw", "litholog"])
        ]
        if candidates:
            return candidates[0], None

    if not AUTO_DOWNLOAD_GIS:
        raise FileNotFoundError(
            "Attach LiMW_GIS-2015.gdb.zip or enable AUTO_DOWNLOAD_GIS."
        )

    archive = download_with_progress(
        GLIM_POLYGON_URL,
        CACHE / "LiMW_GIS-2015.gdb.zip",
    )
    extracted = extract_gis_zip(archive)
    gdbs = list(extracted.rglob("*.gdb"))
    if not gdbs:
        raise FileNotFoundError("Downloaded archive did not produce a .gdb.")
    return gdbs[0], archive

GLIM_SOURCE, GLIM_ARCHIVE = find_vector_source()

print("Polygon source:", GLIM_SOURCE)
if GLIM_ARCHIVE is not None:
    print("Archive:", GLIM_ARCHIVE)
    print("SHA-256:", sha256_file(GLIM_ARCHIVE))


## 3. Inspect polygon layers and identify the GLiM Level 1 field

In [ ]:
GLIM_LEVEL1_DESCRIPTION = {
    "ev": "Evaporites",
    "ig": "Ice and glaciers",
    "mt": "Metamorphics",
    "nd": "No data",
    "pa": "Acid plutonic rocks",
    "pb": "Basic plutonic rocks",
    "pi": "Intermediate plutonic rocks",
    "py": "Pyroclastics",
    "sc": "Carbonate sedimentary rocks",
    "sm": "Mixed sedimentary rocks",
    "ss": "Siliciclastic sedimentary rocks",
    "su": "Unconsolidated sediments",
    "va": "Acid volcanic rocks",
    "vb": "Basic volcanic rocks",
    "vi": "Intermediate volcanic rocks",
    "wb": "Water bodies",
}

DESCRIPTION_TO_CODE = {
    re.sub(r"[^a-z0-9]+", " ", description.lower()).strip(): code
    for code, description in GLIM_LEVEL1_DESCRIPTION.items()
}
DESCRIPTION_TO_CODE.update({
    "metamorphic rock": "mt",
    "metamorphic rocks": "mt",
    "acid plutonics": "pa",
    "basic plutonics": "pb",
    "intermediate plutonics": "pi",
    "acid volcanics": "va",
    "basic volcanics": "vb",
    "intermediate volcanics": "vi",
    "pyroclastic": "py",
    "evaporite": "ev",
    "no data": "nd",
    "water body": "wb",
})

KNOWN_CODES = set(GLIM_LEVEL1_DESCRIPTION)

def value_to_level1_code(value):
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    if not text:
        return None

    normalized = re.sub(r"[^a-z0-9]+", " ", text).strip()
    if normalized in DESCRIPTION_TO_CODE:
        return DESCRIPTION_TO_CODE[normalized]

    match = re.match(
        r"^(ev|ig|mt|nd|pa|pb|pi|py|sc|sm|ss|su|va|vb|vi|wb)",
        text,
    )
    if match:
        return match.group(1)

    tokens = re.findall(r"[a-z]{2}", text)
    recognized = [token for token in tokens if token in KNOWN_CODES]
    if len(recognized) == 1:
        return recognized[0]
    return None

def detect_level1_field(sample, explicit=None):
    if explicit is not None:
        if explicit not in sample.columns:
            raise KeyError(
                f"Requested field '{explicit}' is absent. "
                f"Available: {sample.columns.tolist()}"
            )
        return explicit, 1.0

    preferred = {
        "level1", "level_1", "lithology1", "litho1",
        "lith_class", "lithology", "glim", "code", "xx", "class1",
    }
    preferred_normalized = {
        re.sub(r"[^a-z0-9]+", "", value) for value in preferred
    }

    scores = []
    for column in sample.columns:
        if column == "geometry":
            continue
        values = sample[column].dropna()
        if values.empty:
            continue
        fraction = float(values.astype(str).map(value_to_level1_code).notna().mean())
        name = re.sub(r"[^a-z0-9]+", "", str(column).lower())
        bonus = 0.25 if name in preferred_normalized else 0.0
        scores.append((fraction + bonus, fraction, column))

    if not scores:
        return None, 0.0

    scores.sort(reverse=True)
    _, fraction, column = scores[0]
    return (column, fraction) if fraction >= 0.30 else (None, fraction)

layer_rows = []

for layer_name, geometry_type in pyogrio.list_layers(GLIM_SOURCE):
    try:
        info = pyogrio.read_info(GLIM_SOURCE, layer=layer_name)
        sample = pyogrio.read_dataframe(
            GLIM_SOURCE,
            layer=layer_name,
            max_features=200,
            read_geometry=False,
            use_arrow=True,
        )

        override = None
        if isinstance(LEVEL1_FIELD_OVERRIDES, dict):
            override = LEVEL1_FIELD_OVERRIDES.get(layer_name)
        elif isinstance(LEVEL1_FIELD_OVERRIDES, str):
            override = LEVEL1_FIELD_OVERRIDES

        field, fraction = detect_level1_field(sample, override)
        layer_rows.append({
            "layer": layer_name,
            "geometry_type": geometry_type,
            "feature_count": info.get("features"),
            "crs": info.get("crs"),
            "level1_field": field,
            "detection_fraction": fraction,
            "fields": "|".join(map(str, info.get("fields", []))),
        })
    except Exception as exc:
        layer_rows.append({
            "layer": layer_name,
            "geometry_type": geometry_type,
            "feature_count": None,
            "crs": None,
            "level1_field": None,
            "detection_fraction": 0.0,
            "fields": "",
            "inspection_error": repr(exc),
        })

layer_catalog = pd.DataFrame(layer_rows)
layer_catalog.to_csv(OUT / "glim_layer_catalog.csv", index=False)

polygon_mask = (
    layer_catalog["geometry_type"]
    .fillna("")
    .str.contains("Polygon", case=False, regex=False)
)

candidate_layers = layer_catalog[
    polygon_mask & layer_catalog["level1_field"].notna()
].copy()

if POLYGON_LAYER_OVERRIDES is not None:
    candidate_layers = layer_catalog[
        layer_catalog["layer"].isin(POLYGON_LAYER_OVERRIDES)
    ].copy()

if candidate_layers.empty:
    print(layer_catalog.to_string(index=False))
    raise RuntimeError(
        "No polygon layer with a recognizable Level 1 field was detected. "
        "Inspect glim_layer_catalog.csv, then set the layer/field overrides."
    )

print(candidate_layers[
    ["layer", "geometry_type", "feature_count", "crs", "level1_field"]
].to_string(index=False))


## 4. Predefined cautious mapping

- acid plutonic → `felsic`
- basic plutonic → `mafic`
- acid/intermediate/basic volcanic and pyroclastic → `volcanic`
- carbonate sedimentary → `carbonate`
- siliciclastic sedimentary → `siliciclastic`
- metamorphic → `metamorphic`
- evaporite → `evaporite`

The following remain broad or unresolved:

- intermediate plutonic → `other`
- mixed sedimentary → `other`
- unconsolidated sediment → `other`
- water, ice, and no-data → `unknown`

In [ ]:
GLIM_TO_HGCTM = {
    "ev": ("evaporite", "high", "Direct Level 1 correspondence"),
    "mt": ("metamorphic", "high", "Direct Level 1 correspondence"),
    "pa": ("felsic", "high", "Acid plutonic rock"),
    "pb": ("mafic", "high", "Basic plutonic rock"),
    "py": ("volcanic", "medium", "Pyroclastic surface unit"),
    "sc": ("carbonate", "high", "Carbonate sedimentary rock"),
    "ss": ("siliciclastic", "high", "Siliciclastic sedimentary rock"),
    "va": ("volcanic", "high", "Acid volcanic rock"),
    "vb": ("volcanic", "high", "Basic volcanic rock"),
    "vi": ("volcanic", "high", "Intermediate volcanic rock"),
    "pi": ("other", "low", "Intermediate plutonic not forced into felsic or mafic"),
    "sm": ("other", "low", "Mixed sedimentary not forced into carbonate or siliciclastic"),
    "su": ("other", "low", "Unconsolidated cover not treated as bedrock host lithology"),
    "ig": ("unknown", "none", "Ice/glacier does not identify host lithology"),
    "wb": ("unknown", "none", "Water body does not identify host lithology"),
    "nd": ("unknown", "none", "No GLiM data"),
}

RESOLVED_CLASSES = {
    "felsic", "mafic", "volcanic", "carbonate",
    "siliciclastic", "metamorphic", "evaporite",
}

def map_glim_code(code):
    return GLIM_TO_HGCTM.get(
        code,
        ("unknown", "none", "Missing or unrecognized Level 1 code"),
    )

def lithology_status(value):
    value = str(value).strip().lower()
    if value in RESOLVED_CLASSES:
        return "resolved"
    if value == "other":
        return "other"
    return "unknown"


## 5. Spatial extraction for all 1,335 deposits

In [ ]:
points = gpd.GeoDataFrame(
    cov[["Mindat_id", "Latitude", "Longitude"]].copy(),
    geometry=gpd.points_from_xy(cov["Longitude"], cov["Latitude"]),
    crs="EPSG:4326",
)
points["tile_x"] = np.floor((points["Longitude"] + 180) / TILE_DEGREES).astype(int)
points["tile_y"] = np.floor((points["Latitude"] + 90) / TILE_DEGREES).astype(int)

def safe_field_name(name):
    cleaned = re.sub(r"[^0-9a-zA-Z_]+", "_", str(name)).strip("_").lower()
    return cleaned or "field"

def prefix_original_fields(frame):
    rename = {}
    used = set(frame.columns)
    for column in frame.columns:
        if column == "geometry" or str(column).startswith("__"):
            continue
        base = "glim_attr_" + safe_field_name(column)
        candidate = base
        counter = 2
        while candidate in used or candidate in rename.values():
            candidate = f"{base}_{counter}"
            counter += 1
        rename[column] = candidate
    return frame.rename(columns=rename)

def transform_bounds(bounds, target_crs):
    transformer = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    return transformer.transform_bounds(*bounds, densify_pts=21)

all_matches = []
tile_groups = list(points.groupby(["tile_x", "tile_y"], sort=True))

for layer_row in candidate_layers.to_dict("records"):
    layer = layer_row["layer"]
    level1_field = layer_row["level1_field"]
    layer_crs = layer_row["crs"]

    if not layer_crs:
        raise RuntimeError(f"Layer '{layer}' has no CRS.")

    print(f"\nLayer: {layer}; Level 1 field: {level1_field}")

    for _, point_tile in tqdm(tile_groups, desc=layer):
        bounds_wgs84 = (
            float(point_tile["Longitude"].min() - BBOX_PADDING_DEGREES),
            float(point_tile["Latitude"].min() - BBOX_PADDING_DEGREES),
            float(point_tile["Longitude"].max() + BBOX_PADDING_DEGREES),
            float(point_tile["Latitude"].max() + BBOX_PADDING_DEGREES),
        )
        source_bbox = transform_bounds(bounds_wgs84, layer_crs)

        polygons = pyogrio.read_dataframe(
            GLIM_SOURCE,
            layer=layer,
            bbox=source_bbox,
            use_arrow=True,
        )
        if polygons.empty:
            continue

        polygons = polygons[
            polygons.geometry.notna() & ~polygons.geometry.is_empty
        ].copy()
        if polygons.empty:
            continue

        if polygons.crs is None:
            polygons = polygons.set_crs(layer_crs)

        if level1_field not in polygons.columns:
            raise KeyError(f"Field '{level1_field}' absent in layer '{layer}'.")

        raw_level1 = polygons[level1_field].copy()
        polygons["__glim_level1_value_raw"] = raw_level1.astype(str)
        polygons["__glim_level1_code"] = raw_level1.map(value_to_level1_code)
        polygons["__source_layer"] = layer
        polygons["__source_crs"] = str(polygons.crs)
        polygons["__polygon_row_order"] = np.arange(len(polygons))

        try:
            polygons["__polygon_area_km2"] = (
                polygons.to_crs("EPSG:6933").geometry.area / 1_000_000
            )
        except Exception:
            polygons["__polygon_area_km2"] = np.nan

        polygons = prefix_original_fields(polygons).to_crs("EPSG:4326")

        joined = gpd.sjoin(
            point_tile[["Mindat_id", "geometry"]],
            polygons,
            how="inner",
            predicate="intersects",
        )
        if joined.empty:
            continue

        joined = joined.drop(columns=["geometry"], errors="ignore")
        joined["__original_level1_field"] = level1_field
        all_matches.append(pd.DataFrame(joined))

match_candidates = (
    pd.concat(all_matches, ignore_index=True, sort=False)
    if all_matches
    else pd.DataFrame(columns=["Mindat_id"])
)

print("Match rows:", len(match_candidates))
print("Deposits with at least one exact match:", match_candidates["Mindat_id"].nunique())


In [ ]:
if not match_candidates.empty:
    match_candidates["glim_level1_code"] = match_candidates["__glim_level1_code"]
    match_candidates["glim_level1_description"] = (
        match_candidates["glim_level1_code"].map(GLIM_LEVEL1_DESCRIPTION)
    )

    mapped = match_candidates["glim_level1_code"].map(map_glim_code)
    match_candidates["glim_model_class"] = mapped.map(lambda value: value[0])
    match_candidates["glim_assignment_confidence"] = mapped.map(lambda value: value[1])
    match_candidates["glim_mapping_rationale"] = mapped.map(lambda value: value[2])
    match_candidates["glim_status"] = match_candidates["glim_model_class"].map(lithology_status)

    rank = {"resolved": 0, "other": 1, "unknown": 2}
    match_candidates["__status_rank"] = (
        match_candidates["glim_status"].map(rank).fillna(3)
    )

    overlap = (
        match_candidates.groupby("Mindat_id")
        .agg(
            glim_match_count=("Mindat_id", "size"),
            glim_distinct_code_count=("glim_level1_code", lambda x: x.dropna().nunique()),
            glim_all_match_codes=(
                "glim_level1_code",
                lambda x: "|".join(sorted(set(x.dropna().astype(str)))),
            ),
            glim_all_match_classes=(
                "glim_model_class",
                lambda x: "|".join(sorted(set(x.dropna().astype(str)))),
            ),
            glim_all_match_layers=(
                "__source_layer",
                lambda x: "|".join(sorted(set(x.dropna().astype(str)))),
            ),
        )
        .reset_index()
    )
    overlap["glim_overlap_ambiguous"] = overlap["glim_distinct_code_count"] > 1

    ranked = match_candidates.sort_values(
        [
            "Mindat_id", "__status_rank", "__polygon_area_km2",
            "__source_layer", "__polygon_row_order",
        ],
        na_position="last",
    )
    selected = ranked.drop_duplicates("Mindat_id", keep="first").merge(
        overlap,
        on="Mindat_id",
        how="left",
    )
else:
    selected = pd.DataFrame({"Mindat_id": cov["Mindat_id"]})

selected = cov[["Mindat_id"]].merge(selected, on="Mindat_id", how="left")
selected["glim_match_method"] = np.where(
    selected["glim_level1_code"].notna(),
    "exact_point_in_polygon",
    "no_exact_polygon_match",
)
selected["glim_model_class"] = selected["glim_model_class"].fillna("unknown")
selected["glim_status"] = selected["glim_model_class"].map(lithology_status)
selected["glim_assignment_confidence"] = (
    selected["glim_assignment_confidence"].fillna("none")
)
selected["glim_mapping_rationale"] = selected["glim_mapping_rationale"].fillna(
    "No exact polygon match or unrecognized Level 1 code"
)
selected["glim_match_count"] = selected["glim_match_count"].fillna(0).astype(int)
selected["glim_overlap_ambiguous"] = (
    selected["glim_overlap_ambiguous"].fillna(False)
)
selected["glim_dataset_version"] = GLIM_VERSION_LABEL
selected["glim_vector_source"] = str(GLIM_SOURCE)
selected["glim_extracted_at_utc"] = datetime.now(timezone.utc).isoformat()

print(selected["glim_model_class"].value_counts(dropna=False).to_string())
print("\nStatus:")
print(selected["glim_status"].value_counts(dropna=False).to_string())
print("\nNo exact match:", int((selected["glim_match_count"] == 0).sum()))
print("Ambiguous overlaps:", int(selected["glim_overlap_ambiguous"].sum()))

match_candidates.to_csv(OUT / "glim_all_point_polygon_matches.csv", index=False)
selected.to_csv(OUT / "glim_selected_point_extract_full.csv", index=False)


## 6. Compare GLiM with original Macrostrat

In [ ]:
macro_model = cov[
    [
        "Mindat_id", "Deposit_type", "age_category",
        "litho_class", "Latitude", "Longitude",
    ]
].rename(columns={"litho_class": "macrostrat_class"}).copy()

macro_model["macrostrat_class"] = (
    macro_model["macrostrat_class"].astype(str).str.strip().str.lower()
)
macro_model["macrostrat_status"] = macro_model["macrostrat_class"].map(lithology_status)

macro_raw = macro[
    [
        "Mindat_id", "litho_raw", "litho_class",
        "api_status", "latitude", "longitude",
    ]
].rename(columns={
    "litho_raw": "macrostrat_raw",
    "litho_class": "macrostrat_file_class",
    "api_status": "macrostrat_api_status",
    "latitude": "macrostrat_query_latitude",
    "longitude": "macrostrat_query_longitude",
})

comparison = (
    macro_model
    .merge(macro_raw, on="Mindat_id", how="left", validate="one_to_one")
    .merge(selected, on="Mindat_id", how="left", validate="one_to_one")
)

comparison["both_resolved"] = (
    comparison["macrostrat_status"].eq("resolved")
    & comparison["glim_status"].eq("resolved")
)

def agreement_label(row):
    if row["macrostrat_status"] == "resolved" and row["glim_status"] == "resolved":
        return (
            "both_resolved_agree"
            if row["macrostrat_class"] == row["glim_model_class"]
            else "both_resolved_disagree"
        )
    if row["macrostrat_status"] != "resolved" and row["glim_status"] == "resolved":
        return f"glim_resolves_macrostrat_{row['macrostrat_status']}"
    if row["macrostrat_status"] == "resolved" and row["glim_status"] != "resolved":
        return f"macrostrat_resolved_glim_{row['glim_status']}"
    return (
        f"both_unresolved_macro_{row['macrostrat_status']}"
        f"_glim_{row['glim_status']}"
    )

comparison["agreement_status"] = comparison.apply(agreement_label, axis=1)

both = comparison.loc[comparison["both_resolved"]].copy()
if len(both):
    exact_agreement = float(
        (both["macrostrat_class"] == both["glim_model_class"]).mean()
    )
    kappa = float(
        cohen_kappa_score(
            both["macrostrat_class"],
            both["glim_model_class"],
        )
    )
else:
    exact_agreement = np.nan
    kappa = np.nan

metrics = pd.DataFrame([
    {"metric": "n_model_deposits", "value": len(comparison)},
    {"metric": "n_both_strictly_resolved", "value": len(both)},
    {"metric": "exact_agreement_among_both_resolved", "value": exact_agreement},
    {"metric": "cohen_kappa_among_both_resolved", "value": kappa},
])

confusion = pd.crosstab(
    both["macrostrat_class"],
    both["glim_model_class"],
    margins=True,
)

print(metrics.to_string(index=False))
print("\nConfusion matrix:")
print(confusion.to_string())

comparison.to_csv(OUT / "macrostrat_glim_comparison_full.csv", index=False)
metrics.to_csv(OUT / "macrostrat_glim_agreement_metrics.csv", index=False)
confusion.to_csv(OUT / "macrostrat_glim_confusion_matrix.csv")


## 7. Recovery of original `unknown` and `other` records

In [ ]:
recovery_status = pd.crosstab(
    comparison["macrostrat_status"],
    comparison["glim_status"],
    margins=True,
)

rows = []
for original_status in ["unknown", "other"]:
    subset = comparison[comparison["macrostrat_status"].eq(original_status)]
    resolved = subset["glim_status"].eq("resolved")
    rows.append({
        "macrostrat_group": original_status,
        "n_original": len(subset),
        "n_glim_strictly_resolved": int(resolved.sum()),
        "percent_glim_strictly_resolved": (
            100 * resolved.mean() if len(subset) else np.nan
        ),
        "n_glim_other": int(subset["glim_status"].eq("other").sum()),
        "n_glim_unknown": int(subset["glim_status"].eq("unknown").sum()),
    })

recovery_summary = pd.DataFrame(rows)

recovered_classes = (
    comparison[
        comparison["macrostrat_status"].isin(["unknown", "other"])
        & comparison["glim_status"].eq("resolved")
    ]
    .groupby(["macrostrat_status", "glim_model_class"])
    .size()
    .rename("count")
    .reset_index()
)

print(recovery_status.to_string())
print("\nRecovery summary:")
print(recovery_summary.round(2).to_string(index=False))
print("\nRecovered classes:")
print(recovered_classes.to_string(index=False))

recovery_status.to_csv(OUT / "glim_recovery_status_crosstab.csv")
recovery_summary.to_csv(OUT / "glim_recovery_summary.csv", index=False)
recovered_classes.to_csv(OUT / "glim_recovered_class_distribution.csv", index=False)


## 8. Export three lithology versions

In [ ]:
# A: original Macrostrat
version_a = cov.copy()
version_a["lithology_A_class"] = (
    version_a["litho_class"].astype(str).str.strip().str.lower()
)
version_a["lithology_A_status"] = version_a["lithology_A_class"].map(lithology_status)
version_a["lithology_A_source"] = "Macrostrat_original"

# B: GLiM-only
version_b = cov.drop(columns=["litho_class"]).merge(
    selected[
        [
            "Mindat_id", "glim_level1_code", "glim_level1_description",
            "glim_model_class", "glim_status",
            "glim_assignment_confidence", "glim_mapping_rationale",
            "glim_match_method", "glim_match_count",
            "glim_overlap_ambiguous",
        ]
    ],
    on="Mindat_id",
    how="left",
    validate="one_to_one",
).rename(columns={
    "glim_model_class": "lithology_B_class",
    "glim_status": "lithology_B_status",
    "glim_assignment_confidence": "lithology_B_confidence",
})
version_b["lithology_B_source"] = "GLiM_full_polygon"

# C: Macrostrat with GLiM fallback
def combined_assignment(row):
    if row["macrostrat_status"] == "resolved":
        return (
            row["macrostrat_class"], "resolved", "Macrostrat",
            "original_rule_based", "Macrostrat specific class retained",
        )
    if row["glim_status"] == "resolved":
        return (
            row["glim_model_class"], "resolved", "GLiM_fallback",
            row["glim_assignment_confidence"],
            f"GLiM resolves original Macrostrat {row['macrostrat_status']}",
        )
    if row["macrostrat_status"] == "other" or row["glim_status"] == "other":
        return (
            "other", "other", "unresolved_broad_class", "low",
            "Neither source supports a cautious specific class",
        )
    return (
        "unknown", "unknown", "unresolved_no_class", "none",
        "Neither source provides a usable lithology class",
    )

combined_values = comparison.apply(combined_assignment, axis=1)
combined_table = pd.DataFrame(
    combined_values.tolist(),
    columns=[
        "lithology_C_class", "lithology_C_status",
        "lithology_C_source", "lithology_C_confidence",
        "lithology_C_reason",
    ],
)

version_c = pd.concat(
    [
        comparison[
            [
                "Mindat_id", "Deposit_type", "age_category",
                "Latitude", "Longitude",
                "macrostrat_class", "macrostrat_status",
                "glim_level1_code", "glim_level1_description",
                "glim_model_class", "glim_status",
                "glim_assignment_confidence",
                "agreement_status", "glim_overlap_ambiguous",
            ]
        ].reset_index(drop=True),
        combined_table.reset_index(drop=True),
    ],
    axis=1,
)

print("A — Macrostrat:")
print(version_a["lithology_A_status"].value_counts().to_string())
print("\nB — GLiM:")
print(version_b["lithology_B_status"].value_counts().to_string())
print("\nC — Combined:")
print(version_c["lithology_C_status"].value_counts().to_string())

version_a.to_csv(
    OUT / "covariates_lithology_A_macrostrat_original.csv",
    index=False,
)
version_b.to_csv(
    OUT / "covariates_lithology_B_glim_only.csv",
    index=False,
)
version_c.to_csv(
    OUT / "covariates_lithology_C_macrostrat_glim_combined.csv",
    index=False,
)


## 9. Manual deposit-scale host-rock validation list

In [ ]:
def assign_continent_audit_only(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return "Unknown"
    if lat > 15 and -170 < lon < -50:
        return "North_America"
    if lat <= 15 and -120 < lon < -30:
        return "South_America"
    if lat > 35 and -25 < lon < 60:
        return "Europe"
    if lat > 0 and 60 < lon < 180:
        return "Asia"
    if lat <= 0 and 95 < lon < 180:
        return "Oceania"
    if -40 < lat < 40 and -25 < lon < 55:
        return "Africa"
    if lat <= -10 and 110 < lon < 180:
        return "Oceania"
    return "Other"

def validation_group(row):
    if row["macrostrat_status"] == "unknown" and row["glim_status"] == "resolved":
        return "1_macro_unknown_glim_resolved"
    if row["macrostrat_status"] == "other" and row["glim_status"] == "resolved":
        return "2_macro_other_glim_resolved"
    if row["agreement_status"] == "both_resolved_disagree":
        return "3_both_resolved_disagree"
    if row["macrostrat_status"] != "resolved" and row["glim_status"] != "resolved":
        return "4_both_unresolved"
    if row["macrostrat_status"] == "resolved" and row["glim_status"] != "resolved":
        return "5_macro_only_resolved"
    return "6_both_resolved_agree"

validation = comparison.copy()
validation["continent_approx_audit_only"] = validation.apply(
    lambda row: assign_continent_audit_only(row["Latitude"], row["Longitude"]),
    axis=1,
)
validation["validation_group"] = validation.apply(validation_group, axis=1)
validation["validation_priority"] = (
    validation["validation_group"].str.extract(r"^(\d+)")[0].astype(int)
)

if deposit_lookup is not None:
    validation = validation.merge(
        deposit_lookup,
        on="Mindat_id",
        how="left",
        suffixes=("", "_main_database"),
        validate="one_to_one",
    )

validation = validation.sort_values(
    [
        "validation_priority", "Deposit_type",
        "continent_approx_audit_only", "Mindat_id",
    ]
).reset_index(drop=True)

def balanced_sample(frame, n, seed):
    if n <= 0 or frame.empty:
        return frame.iloc[0:0].copy()
    if len(frame) <= n:
        return frame.copy()

    rng = np.random.default_rng(seed)
    work = frame.copy()
    work["__random"] = rng.random(len(work))
    work["__stratum"] = (
        work["Deposit_type"].fillna("Unknown").astype(str)
        + "|"
        + work["continent_approx_audit_only"].fillna("Unknown").astype(str)
    )
    work = work.sort_values("__random")

    strata = {
        key: list(group.index)
        for key, group in work.groupby("__stratum", sort=True)
    }
    selected_indices = []

    while len(selected_indices) < n and any(strata.values()):
        for key in sorted(strata):
            if strata[key] and len(selected_indices) < n:
                selected_indices.append(strata[key].pop(0))

    return work.loc[selected_indices].drop(
        columns=["__random", "__stratum"],
        errors="ignore",
    )

quotas = {
    "1_macro_unknown_glim_resolved": 30,
    "2_macro_other_glim_resolved": 30,
    "3_both_resolved_disagree": 30,
    "4_both_unresolved": 10,
    "5_macro_only_resolved": 10,
    "6_both_resolved_agree": 10,
}

sample_parts = []
for offset, (group_name, quota) in enumerate(quotas.items()):
    group_frame = validation[validation["validation_group"].eq(group_name)]
    sample_parts.append(
        balanced_sample(
            group_frame,
            min(quota, len(group_frame)),
            RANDOM_SEED + offset,
        )
    )

validation_sample = (
    pd.concat(sample_parts, ignore_index=True, sort=False)
    .drop_duplicates("Mindat_id")
)

shortfall = MANUAL_VALIDATION_SAMPLE_N - len(validation_sample)
if shortfall > 0:
    remaining = validation[
        ~validation["Mindat_id"].isin(validation_sample["Mindat_id"])
    ]
    fill = balanced_sample(
        remaining,
        min(shortfall, len(remaining)),
        RANDOM_SEED + 100,
    )
    validation_sample = pd.concat(
        [validation_sample, fill],
        ignore_index=True,
        sort=False,
    )

validation_sample = validation_sample.sort_values(
    ["validation_priority", "Deposit_type", "Mindat_id"]
).reset_index(drop=True)

print("All deposits:")
print(validation["validation_group"].value_counts().to_string())
print("\nRecommended validation sample:")
print(validation_sample["validation_group"].value_counts().to_string())
print("\nSample size:", len(validation_sample))

validation.to_csv(
    OUT / "manual_host_lithology_validation_priority_all.csv",
    index=False,
)
validation_sample.to_csv(
    OUT / "manual_host_lithology_validation_sample_120.csv",
    index=False,
)


## 10. Save provenance and create one downloadable archive

In [ ]:
metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "HGCTM_Phase1B_GLiM_Lithology_Audit.ipynb",
    "model_deposit_count": int(len(cov)),
    "stage0_directory": str(STAGE0),
    "polygon_source_path": str(GLIM_SOURCE),
    "polygon_archive_path": str(GLIM_ARCHIVE) if GLIM_ARCHIVE else None,
    "polygon_archive_sha256": (
        sha256_file(GLIM_ARCHIVE)
        if GLIM_ARCHIVE is not None and Path(GLIM_ARCHIVE).is_file()
        else None
    ),
    "polygon_download_url": GLIM_POLYGON_URL,
    "polygon_version_label": GLIM_VERSION_LABEL,
    "selected_layers": candidate_layers[
        ["layer", "geometry_type", "feature_count", "crs", "level1_field"]
    ].to_dict("records"),
    "spatial_match": {
        "method": "exact point in polygon",
        "tile_degrees": TILE_DEGREES,
        "bbox_padding_degrees": BBOX_PADDING_DEGREES,
        "nearest_polygon_fallback": False,
        "overlap_rule": [
            "prefer resolved over other/unknown",
            "prefer smallest polygon",
            "use layer and row order only as deterministic tie-breakers",
        ],
    },
    "mapping": {
        code: {
            "description": GLIM_LEVEL1_DESCRIPTION[code],
            "hgctm_class": GLIM_TO_HGCTM[code][0],
            "confidence": GLIM_TO_HGCTM[code][1],
            "rationale": GLIM_TO_HGCTM[code][2],
        }
        for code in GLIM_LEVEL1_DESCRIPTION
    },
    "references": [
        {
            "authors": "Jens Hartmann; Nils Moosdorf",
            "title": (
                "The new global lithological map database GLiM: "
                "A representation of rock properties at the Earth surface"
            ),
            "doi": "10.1029/2012GC004370",
        },
        {
            "authors": "Jens Hartmann; Nils Moosdorf",
            "title": (
                "Global Lithological Map Database v1.0 "
                "(gridded to 0.5° spatial resolution)"
            ),
            "doi": "10.1594/PANGAEA.788537",
            "note": "Coarse gridded product not used in this notebook.",
        },
    ],
}

with open(
    OUT / "phase1b_source_and_mapping_metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(metadata, handle, indent=2, ensure_ascii=False)

readme_lines = [
    "HGCTM Phase 1B GLiM Lithology Audit",
    f"Created: {metadata['created_at_utc']}",
    "",
    "This archive contains:",
    "- full selected GLiM point extraction;",
    "- all exact point-polygon matches;",
    "- Macrostrat-GLiM comparison and agreement metrics;",
    "- recovery summaries for original unknown and other records;",
    "- three lithology covariate versions;",
    "- ranked and sampled manual host-lithology validation lists;",
    "- source, layer, mapping, and checksum metadata.",
    "",
    "No HGCTM-S model was refitted.",
    "Both sources remain surface/map-unit proxies and require",
    "deposit-scale validation for a representative subset.",
]
(OUT / "README.txt").write_text(
    "\n".join(readme_lines),
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(WORK / "HGCTM_Phase1B_GLiM_Audit_Outputs"),
    "zip",
    root_dir=OUT,
)

print("Output files:")
for path in sorted(OUT.iterdir()):
    print(" -", path.name)

print("\nDownload from Kaggle Output:")
print(archive_path)
